In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)



In [ ]:
# Task 2: Write your code here:

print(f"Dataset shape: {df_delivery.shape}")
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution ')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_delivery = df_delivery.drop(columns="Order_ID", axis=1)
df_delivery.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery)

In [ ]:
#Task 2 :
df_delivery['Weather'] = df_delivery['Weather'].fillna('none')  # Beacuse it is categorical
df_delivery['Traffic_Level'] = df_delivery['Traffic_Level'].fillna('none')
df_delivery['Time_of_Day'] = df_delivery['Time_of_Day'].fillna('none')

df_delivery['Courier_Experience_yrs'] = df_delivery['Courier_Experience_yrs'].fillna(df_delivery['Courier_Experience_yrs'].mean()) # fill numrical with mean
df_delivery['Delivery_Time'] = df_delivery['Delivery_Time'].fillna(df_delivery['Delivery_Time'].mean())

check_missing_values(df_delivery)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_delivery.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
    le = LabelEncoder()
    df_delivery[col] = le.fit_transform(df_delivery[col].astype(str))

df_delivery.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_delivery.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_delivery[numerical_cols] = scaler.fit_transform(df_delivery[numerical_cols])
df_delivery.head()


In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_delivery, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_delivery.drop("Delivery_Time",axis=1).astype(float)
y = df_delivery['Delivery_Time'].astype(float)

In [ ]:
import tqdm
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm.tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score

n_splits=5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X,y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
    y_pred = np.dot(X_test.values, theta)

    #evalute
    mae = mean_absolute_error(y_test, y_pred)
    lr_losses.append(losses)
    lr_mae.append(mae)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X, y)
print("Model trained!")




In [ ]:
feature_cols = list(df_delivery.columns)
print(feature_cols)

In [ ]:
# Task 1: Write your code here:
feature_cols = list(df_delivery.columns)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: